# Kdr-Hi — OOD Holdout vs Random Shuffle Protocol Comparison

This notebook compares two inner hyperparameter-selection protocols on the kdr-Hi task:

1. **OOD holdout**: the validation subset is chemically separated from the inner training subset.
2. **Random shuffle**: the validation subset is obtained by randomly splitting the outer training set.

The main experimental question is:

> Does in-distribution hyperparameter selection produce higher internal validation scores without improving final out-of-distribution test performance?

The analysis focuses on:

- inner validation PR-AUC;
- inner train PR-AUC;
- outer train PR-AUC after refitting;
- final OOD test PR-AUC;
- inner-to-test generalization gap;
- train-to-test generalization gap;
- selected hyperparameters across folds.

The models considered are:

- Decision Tree;
- Logistic Regression;
- Linear SVM.

The fingerprints considered are:

- ECFP4;
- MACCS;
- RDKit descriptors, where available.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("../..").resolve()
RESULTS_ROOT = PROJECT_ROOT / "results" / "hi" / "kdr"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"RESULTS_ROOT exists: {RESULTS_ROOT.exists()}")

PROJECT_ROOT: /home/f.capria/drug-discovery-lohi
RESULTS_ROOT: /home/f.capria/drug-discovery-lohi/results/hi/kdr
RESULTS_ROOT exists: True


# Experiment registry

In [2]:
EXPERIMENTS = [
    # ---------------------------------------------------------------------
    # Decision Tree
    # ---------------------------------------------------------------------
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_kdr_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_kdr_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_kdr_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_kdr_hi_random_shuffle_maccs",
    },

    # ---------------------------------------------------------------------
    # Logistic Regression
    # ---------------------------------------------------------------------
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_kdr_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_kdr_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_kdr_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_kdr_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_kdr_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_kdr_hi_random_shuffle_rdkit_desc",
    },

    # ---------------------------------------------------------------------
    # Linear SVM
    # ---------------------------------------------------------------------
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_kdr_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_kdr_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_kdr_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_kdr_hi_random_shuffle_maccs",
    },
]

registry = pd.DataFrame(EXPERIMENTS)
registry["path"] = registry["result_dir"].apply(lambda d: RESULTS_ROOT / d)
registry["exists"] = registry["path"].apply(lambda p: p.exists())

registry

,model,model_short,fingerprint,protocol,result_dir,path,exists
0,Decision Tree,DT,ECFP4,OOD holdout,dt_kdr_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/kdr/dt_kdr_hi_inner_ood_holdout_ecfp4,True
1,Decision Tree,DT,MACCS,OOD holdout,dt_kdr_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/kdr/dt_kdr_hi_inner_ood_holdout_maccs,True
2,Decision Tree,DT,ECFP4,Random shuffle,dt_kdr_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/kdr/dt_kdr_hi_random_shuffle_ecfp4,True
3,Decision Tree,DT,MACCS,Random shuffle,dt_kdr_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/kdr/dt_kdr_hi_random_shuffle_maccs,True
4,Logistic Regression,LR,ECFP4,OOD holdout,lr_kdr_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/kdr/lr_kdr_hi_inner_ood_holdout_ecfp4,True
5,Logistic Regression,LR,MACCS,OOD holdout,lr_kdr_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/kdr/lr_kdr_hi_inner_ood_holdout_maccs,True
6,Logistic Regression,LR,RDKit desc,OOD holdout,lr_kdr_hi_inner_ood_holdout_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/kdr/lr_kdr_hi_inner_ood_holdout_rdkit_desc,True
7,Logistic Regression,LR,ECFP4,Random shuffle,lr_kdr_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/kdr/lr_kdr_hi_random_shuffle_ecfp4,True
8,Logistic Regression,LR,MACCS,Random shuffle,lr_kdr_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/kdr/lr_kdr_hi_random_shuffle_maccs,True
9,Logistic Regression,LR,RDKit desc,Random shuffle,lr_kdr_hi_random_shuffle_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/kdr/lr_kdr_hi_random_shuffle_rdkit_desc,True


In [3]:
missing = registry.loc[~registry["exists"], ["model", "fingerprint", "protocol", "path"]]

if len(missing) > 0:
    print("Missing result folders:")
    display(missing)
else:
    print("All registered result folders exist.")

All registered result folders exist.


# Load params_fold_i.json into df_folds

In [4]:
def load_params_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def safe_get_metric(metrics: dict, key: str):
    if metrics is None:
        return np.nan
    return metrics.get(key, np.nan)


rows = []

for exp in EXPERIMENTS:
    result_path = RESULTS_ROOT / exp["result_dir"]

    for fold in [1, 2, 3]:
        params_path = result_path / f"params_fold_{fold}.json"

        if not params_path.exists():
            print(f"Missing file: {params_path}")
            continue

        data = load_params_json(params_path)

        train_metrics = data.get("train_metrics", {})
        test_metrics = data.get("test_metrics", {})
        best_params = data.get("best_params", {})

        inner_pr_auc = data.get("inner_selection_score", np.nan)
        inner_train_pr_auc = data.get("inner_train_score", np.nan)
        train_pr_auc = safe_get_metric(train_metrics, "pr_auc")
        test_pr_auc = safe_get_metric(test_metrics, "pr_auc")

        row = {
            "model": exp["model"],
            "model_short": exp["model_short"],
            "fingerprint": exp["fingerprint"],
            "protocol": exp["protocol"],
            "result_dir": exp["result_dir"],
            "fold": fold,

            "inner_pr_auc": inner_pr_auc,
            "inner_train_pr_auc": inner_train_pr_auc,
            "train_pr_auc": train_pr_auc,
            "test_pr_auc": test_pr_auc,

            "inner_test_gap": inner_pr_auc - test_pr_auc,
            "train_test_gap": train_pr_auc - test_pr_auc,

            "best_params": best_params,
            "inner_split_strategy": data.get("inner_split_strategy", None),
            "time_seconds": data.get("time_seconds", np.nan),
        }

        rows.append(row)

df_folds = pd.DataFrame(rows)

order_model = {
    "Decision Tree": 0,
    "Logistic Regression": 1,
    "Linear SVM": 2,
}

order_fp = {
    "ECFP4": 0,
    "MACCS": 1,
    "RDKit desc": 2,
}

order_protocol = {
    "OOD holdout": 0,
    "Random shuffle": 1,
}

df_folds["model_order"] = df_folds["model"].map(order_model)
df_folds["fingerprint_order"] = df_folds["fingerprint"].map(order_fp)
df_folds["protocol_order"] = df_folds["protocol"].map(order_protocol)

df_folds = df_folds.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

print(f"Loaded rows: {len(df_folds)}")
df_folds.head(2)

Loaded rows: 42


,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,best_params,inner_split_strategy,time_seconds,model_order,fingerprint_order,protocol_order
0,Decision Tree,DT,ECFP4,OOD holdout,dt_kdr_hi_inner_ood_holdout_ecfp4,1,0.862853,0.951132,0.9783,0.9105,-0.047647,0.0678,"{'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 20, 'max_features': 'log2', 'min...",holdout,18.9,0,0,0
1,Decision Tree,DT,ECFP4,OOD holdout,dt_kdr_hi_inner_ood_holdout_ecfp4,2,0.785762,0.953923,0.5579,0.9438,-0.158038,-0.3859,"{'ccp_alpha': 0.001, 'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'm...",holdout,16.0,0,0,0


# Per-fold table

In [5]:
per_fold_table = df_folds[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "inner_train_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
    ]
].copy()

numeric_cols = [
    "inner_pr_auc",
    "inner_train_pr_auc",
    "train_pr_auc",                                                                                         
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

per_fold_table[numeric_cols] = per_fold_table[numeric_cols].round(4)

per_fold_table

,model,fingerprint,protocol,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.8629,0.9511,0.9783,0.9105,-0.0476,0.0678
1,Decision Tree,ECFP4,OOD holdout,2,0.7858,0.9539,0.5579,0.9438,-0.1580,-0.3859
2,Decision Tree,ECFP4,OOD holdout,3,0.9023,0.9823,0.9870,0.9726,-0.0703,0.0144
3,Decision Tree,ECFP4,Random shuffle,1,0.9932,0.9932,0.9925,0.5623,0.4309,0.4302
4,Decision Tree,ECFP4,Random shuffle,2,0.4045,0.6710,0.5802,0.7389,-0.3344,-0.1587
5,Decision Tree,ECFP4,Random shuffle,3,0.9708,0.9834,0.9797,0.6454,0.3254,0.3343
6,Decision Tree,MACCS,OOD holdout,1,0.8808,0.9494,0.9918,0.9490,-0.0682,0.0428
7,Decision Tree,MACCS,OOD holdout,2,0.8151,0.9453,0.5716,0.9524,-0.1373,-0.3808
8,Decision Tree,MACCS,OOD holdout,3,0.8550,0.9657,0.9902,0.9355,-0.0805,0.0547
9,Decision Tree,MACCS,Random shuffle,1,0.9692,0.9828,0.9769,0.5277,0.4415,0.4492


# Aggregated summary table

In [6]:
def mean_std_string(mean_value, std_value, digits=4):
    if pd.isna(mean_value):
        return ""
    if pd.isna(std_value):
        return f"{mean_value:.{digits}f}"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"


summary_numeric = (
    df_folds
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    .agg(
        inner_mean=("inner_pr_auc", "mean"),
        inner_std=("inner_pr_auc", "std"),
        inner_train_mean=("inner_train_pr_auc", "mean"),
        inner_train_std=("inner_train_pr_auc", "std"),
        train_mean=("train_pr_auc", "mean"),
        train_std=("train_pr_auc", "std"),
        test_mean=("test_pr_auc", "mean"),
        test_std=("test_pr_auc", "std"),
        inner_test_gap_mean=("inner_test_gap", "mean"),
        inner_test_gap_std=("inner_test_gap", "std"),
        train_test_gap_mean=("train_test_gap", "mean"),
        train_test_gap_std=("train_test_gap", "std"),
    )
)

summary_numeric["model_order"] = summary_numeric["model"].map(order_model)
summary_numeric["fingerprint_order"] = summary_numeric["fingerprint"].map(order_fp)
summary_numeric["protocol_order"] = summary_numeric["protocol"].map(order_protocol)

summary_numeric = summary_numeric.sort_values(
    ["model_order", "fingerprint_order", "protocol_order"]
).reset_index(drop=True)

summary_table = summary_numeric[
    ["model", "fingerprint", "protocol"]
].copy()

summary_table["Inner PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_mean"], r["inner_std"]), axis=1
)

summary_table["Inner train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_train_mean"], r["inner_train_std"]), axis=1
)

summary_table["Train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_mean"], r["train_std"]), axis=1
)

summary_table["Final OOD test PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["test_mean"], r["test_std"]), axis=1
)

summary_table["Inner-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_test_gap_mean"], r["inner_test_gap_std"]), axis=1
)

summary_table["Train-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_test_gap_mean"], r["train_test_gap_std"]), axis=1
)

summary_table

,model,fingerprint,protocol,Inner PR-AUC,Inner train PR-AUC,Train PR-AUC,Final OOD test PR-AUC,Inner-test gap,Train-test gap
0,Decision Tree,ECFP4,OOD holdout,0.8503 ± 0.0593,0.9625 ± 0.0172,0.8411 ± 0.2453,0.9423 ± 0.0311,-0.0920 ± 0.0583,-0.1012 ± 0.2480
1,Decision Tree,ECFP4,Random shuffle,0.7895 ± 0.3336,0.8825 ± 0.1832,0.8508 ± 0.2344,0.6489 ± 0.0884,0.1406 ± 0.4147,0.2019 ± 0.3160
2,Decision Tree,MACCS,OOD holdout,0.8503 ± 0.0331,0.9535 ± 0.0108,0.8512 ± 0.2421,0.9456 ± 0.0089,-0.0953 ± 0.0369,-0.0944 ± 0.2481
3,Decision Tree,MACCS,Random shuffle,0.7837 ± 0.3232,0.8394 ± 0.2589,0.8157 ± 0.2954,0.6062 ± 0.1052,0.1775 ± 0.4271,0.2095 ± 0.3989
4,Logistic Regression,ECFP4,OOD holdout,0.8676 ± 0.0521,0.9726 ± 0.0042,0.8698 ± 0.2087,0.9563 ± 0.0109,-0.0887 ± 0.0629,-0.0865 ± 0.2188
5,Logistic Regression,ECFP4,Random shuffle,0.7793 ± 0.3457,0.9677 ± 0.0403,0.9677 ± 0.0388,0.6664 ± 0.0633,0.1129 ± 0.3915,0.3013 ± 0.0963
6,Logistic Regression,MACCS,OOD holdout,0.7873 ± 0.0539,0.8880 ± 0.0069,0.6962 ± 0.4197,0.8431 ± 0.0374,-0.0558 ± 0.0852,-0.1469 ± 0.4561
7,Logistic Regression,MACCS,Random shuffle,0.7681 ± 0.3554,0.8125 ± 0.2895,0.8152 ± 0.2844,0.6105 ± 0.1136,0.1577 ± 0.4689,0.2048 ± 0.3980
8,Logistic Regression,RDKit desc,OOD holdout,0.8073 ± 0.0440,0.9185 ± 0.0070,0.7336 ± 0.3845,0.8776 ± 0.0295,-0.0702 ± 0.0623,-0.1440 ± 0.4098
9,Logistic Regression,RDKit desc,Random shuffle,0.8000 ± 0.3022,0.9535 ± 0.0556,0.9469 ± 0.0642,0.6578 ± 0.0788,0.1421 ± 0.3788,0.2891 ± 0.1400


# Delta table

In [7]:
pivot_summary = summary_numeric.pivot_table(
    index=["model", "model_short", "fingerprint"],
    columns="protocol",
    values=[
        "inner_mean",
        "test_mean",
        "inner_test_gap_mean",
        "train_test_gap_mean",
    ],
)

delta_rows = []

for idx, row in pivot_summary.iterrows():
    model, model_short, fingerprint = idx

    try:
        random_inner = row[("inner_mean", "Random shuffle")]
        ood_inner = row[("inner_mean", "OOD holdout")]

        random_test = row[("test_mean", "Random shuffle")]
        ood_test = row[("test_mean", "OOD holdout")]

        random_inner_gap = row[("inner_test_gap_mean", "Random shuffle")]
        ood_inner_gap = row[("inner_test_gap_mean", "OOD holdout")]

        random_train_gap = row[("train_test_gap_mean", "Random shuffle")]
        ood_train_gap = row[("train_test_gap_mean", "OOD holdout")]

    except KeyError:
        continue

    delta_rows.append({
        "model": model,
        "model_short": model_short,
        "fingerprint": fingerprint,

        "ood_inner_mean": ood_inner,
        "random_inner_mean": random_inner,
        "delta_inner": random_inner - ood_inner,

        "ood_test_mean": ood_test,
        "random_test_mean": random_test,
        "delta_test": random_test - ood_test,

        "ood_inner_test_gap": ood_inner_gap,
        "random_inner_test_gap": random_inner_gap,
        "delta_inner_test_gap": random_inner_gap - ood_inner_gap,

        "ood_train_test_gap": ood_train_gap,
        "random_train_test_gap": random_train_gap,
        "delta_train_test_gap": random_train_gap - ood_train_gap,
    })

delta_table = pd.DataFrame(delta_rows)

delta_table["model_order"] = delta_table["model"].map(order_model)
delta_table["fingerprint_order"] = delta_table["fingerprint"].map(order_fp)

delta_table = delta_table.sort_values(
    ["model_order", "fingerprint_order"]
).reset_index(drop=True)

delta_display = delta_table[
    [
        "model",
        "fingerprint",
        "ood_inner_mean",
        "random_inner_mean",
        "delta_inner",
        "ood_test_mean",
        "random_test_mean",
        "delta_test",
        "ood_inner_test_gap",
        "random_inner_test_gap",
        "delta_inner_test_gap",
        "delta_train_test_gap",
    ]
].copy()

delta_numeric_cols = delta_display.select_dtypes(include=[np.number]).columns
delta_display[delta_numeric_cols] = delta_display[delta_numeric_cols].round(4)

delta_display

,model,fingerprint,ood_inner_mean,random_inner_mean,delta_inner,ood_test_mean,random_test_mean,delta_test,ood_inner_test_gap,random_inner_test_gap,delta_inner_test_gap,delta_train_test_gap
0,Decision Tree,ECFP4,0.8503,0.7895,-0.0608,0.9423,0.6489,-0.2934,-0.0920,0.1406,0.2326,0.3032
1,Decision Tree,MACCS,0.8503,0.7837,-0.0666,0.9456,0.6062,-0.3394,-0.0953,0.1775,0.2728,0.3039
2,Logistic Regression,ECFP4,0.8676,0.7793,-0.0883,0.9563,0.6664,-0.2899,-0.0887,0.1129,0.2016,0.3878
3,Logistic Regression,MACCS,0.7873,0.7681,-0.0192,0.8431,0.6105,-0.2326,-0.0558,0.1577,0.2135,0.3517
4,Logistic Regression,RDKit desc,0.8073,0.8000,-0.0074,0.8776,0.6578,-0.2197,-0.0702,0.1421,0.2124,0.4331
5,Linear SVM,ECFP4,0.8682,0.7864,-0.0818,0.9440,0.6943,-0.2497,-0.0759,0.0920,0.1679,0.3323
6,Linear SVM,MACCS,0.7849,0.7659,-0.0190,0.8339,0.5840,-0.2499,-0.0490,0.1819,0.2309,0.3499


# Hyperparameter summary tables

In [8]:
def flatten_best_params(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in df.iterrows():
        base = {
            "model": row["model"],
            "fingerprint": row["fingerprint"],
            "protocol": row["protocol"],
            "fold": row["fold"],
            "inner_pr_auc": row["inner_pr_auc"],
            "test_pr_auc": row["test_pr_auc"],
        }

        params = row["best_params"]

        if isinstance(params, dict):
            for key, value in params.items():
                clean_key = key.replace("model__", "")
                base[clean_key] = value

        rows.append(base)

    return pd.DataFrame(rows)


hp_df = flatten_best_params(df_folds)

hp_df["model_order"] = hp_df["model"].map(order_model)
hp_df["fingerprint_order"] = hp_df["fingerprint"].map(order_fp)
hp_df["protocol_order"] = hp_df["protocol"].map(order_protocol)

hp_df = hp_df.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

for col in ["inner_pr_auc", "test_pr_auc"]:
    hp_df[col] = hp_df[col].round(4)

hp_df.head()

,model,fingerprint,protocol,fold,inner_pr_auc,test_pr_auc,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio,model_order,fingerprint_order,protocol_order
0,Decision Tree,ECFP4,OOD holdout,1,0.8629,0.9105,0.000,balanced,entropy,20.0,log2,1.0,5.0,NaN,NaN,0,0,0
1,Decision Tree,ECFP4,OOD holdout,2,0.7858,0.9438,0.001,balanced,entropy,20.0,sqrt,2.0,2.0,NaN,NaN,0,0,0
2,Decision Tree,ECFP4,OOD holdout,3,0.9023,0.9726,0.000,NaN,entropy,20.0,NaN,2.0,20.0,NaN,NaN,0,0,0
3,Decision Tree,ECFP4,Random shuffle,1,0.9932,0.5623,0.000,balanced,entropy,20.0,log2,1.0,10.0,NaN,NaN,0,0,1
4,Decision Tree,ECFP4,Random shuffle,2,0.4045,0.7389,0.010,NaN,entropy,5.0,NaN,2.0,2.0,NaN,NaN,0,0,1


# Logistic Regression hyperparameters

In [9]:
lr_hp_table = hp_df.loc[
    hp_df["model"] == "Logistic Regression",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "l1_ratio",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

lr_hp_table = lr_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

lr_hp_table

,protocol,fingerprint,fold,C,class_weight,l1_ratio,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,0.100,balanced,0.00,0.8841,0.9544
1,OOD holdout,ECFP4,2,0.100,balanced,0.00,0.8093,0.9680
2,OOD holdout,ECFP4,3,0.100,NaN,0.00,0.9095,0.9465
3,Random shuffle,ECFP4,1,0.100,NaN,0.00,0.9899,0.5947
4,Random shuffle,ECFP4,2,1.000,NaN,0.25,0.3803,0.7144
5,Random shuffle,ECFP4,3,0.005,NaN,0.00,0.9678,0.6901
6,OOD holdout,MACCS,1,0.100,balanced,0.25,0.8408,0.8307
7,OOD holdout,MACCS,2,0.500,balanced,0.00,0.7330,0.8851
8,OOD holdout,MACCS,3,0.050,NaN,0.50,0.7880,0.8135
9,Random shuffle,MACCS,1,1.000,NaN,1.00,0.9701,0.5441


# Svm liner hyperparameters

In [10]:
svm_hp_table = hp_df.loc[
    hp_df["model"] == "Linear SVM",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

svm_hp_table = svm_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

svm_hp_table

,protocol,fingerprint,fold,C,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,0.050,balanced,0.8853,0.9393
1,OOD holdout,ECFP4,2,0.050,balanced,0.8166,0.9548
2,OOD holdout,ECFP4,3,0.050,balanced,0.9026,0.9380
3,Random shuffle,ECFP4,1,0.010,NaN,0.9936,0.6479
4,Random shuffle,ECFP4,2,0.010,NaN,0.3974,0.7522
5,Random shuffle,ECFP4,3,0.001,balanced,0.9682,0.6829
6,OOD holdout,MACCS,1,0.050,balanced,0.8364,0.8285
7,OOD holdout,MACCS,2,50.000,balanced,0.7330,0.8734
8,OOD holdout,MACCS,3,0.100,NaN,0.7853,0.7998
9,Random shuffle,MACCS,1,2.000,NaN,0.9722,0.5052


# Decision Tree hyperparameters

In [11]:
dt_hp_table = hp_df.loc[
    hp_df["model"] == "Decision Tree",
    [
        "protocol",
        "fingerprint",
        "fold",
        "criterion",
        "max_depth",
        "max_features",
        "min_samples_leaf",
        "min_samples_split",
        "ccp_alpha",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

dt_hp_table = dt_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

dt_hp_table

,protocol,fingerprint,fold,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,ccp_alpha,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,entropy,20.0,log2,1.0,5.0,0.0000,balanced,0.8629,0.9105
1,OOD holdout,ECFP4,2,entropy,20.0,sqrt,2.0,2.0,0.0010,balanced,0.7858,0.9438
2,OOD holdout,ECFP4,3,entropy,20.0,NaN,2.0,20.0,0.0000,NaN,0.9023,0.9726
3,Random shuffle,ECFP4,1,entropy,20.0,log2,1.0,10.0,0.0000,balanced,0.9932,0.5623
4,Random shuffle,ECFP4,2,entropy,5.0,NaN,2.0,2.0,0.0100,NaN,0.4045,0.7389
5,Random shuffle,ECFP4,3,gini,20.0,log2,2.0,5.0,0.0000,balanced,0.9708,0.6454
6,OOD holdout,MACCS,1,entropy,15.0,NaN,10.0,2.0,0.0000,balanced,0.8808,0.9490
7,OOD holdout,MACCS,2,entropy,10.0,NaN,10.0,2.0,0.0000,balanced,0.8151,0.9524
8,OOD holdout,MACCS,3,entropy,20.0,sqrt,2.0,20.0,0.0001,NaN,0.8550,0.9355
9,Random shuffle,MACCS,1,gini,10.0,sqrt,1.0,20.0,0.0000,NaN,0.9692,0.5277


# Compact hyperparameter set summary

In [12]:
def unique_values_as_string(series: pd.Series) -> str:
    values = []
    for value in series.dropna().tolist():
        if value not in values:
            values.append(value)
    return "{" + ", ".join(str(v) for v in values) + "}"


hp_set_summary_rows = []

for model_name, model_df in hp_df.groupby("model"):
    param_cols = [
        c for c in model_df.columns
        if c not in [
            "model",
            "fingerprint",
            "protocol",
            "fold",
            "inner_pr_auc",
            "test_pr_auc",
            "model_order",
            "fingerprint_order",
            "protocol_order",
        ]
    ]

    for protocol, protocol_df in model_df.groupby("protocol"):
        row = {
            "model": model_name,
            "protocol": protocol,
        }

        for col in param_cols:
            if protocol_df[col].notna().any():
                row[col] = unique_values_as_string(protocol_df[col])

        hp_set_summary_rows.append(row)

hp_set_summary = pd.DataFrame(hp_set_summary_rows)
hp_set_summary["model_order"] = hp_set_summary["model"].map(order_model)
hp_set_summary["protocol_order"] = hp_set_summary["protocol"].map(order_protocol)

hp_set_summary = hp_set_summary.sort_values(
    ["model_order", "protocol_order"]
).drop(columns=["model_order", "protocol_order"]).reset_index(drop=True)

hp_set_summary

,model,protocol,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio
0,Decision Tree,OOD holdout,"{0.0, 0.001, 0.0001}",{balanced},{entropy},"{20.0, 15.0, 10.0}","{log2, sqrt}","{1.0, 2.0, 10.0}","{5.0, 2.0, 20.0}",NaN,NaN
1,Decision Tree,Random shuffle,"{0.0, 0.01, 0.001}",{balanced},"{entropy, gini}","{20.0, 5.0, 10.0, 15.0}","{log2, sqrt}","{1.0, 2.0}","{10.0, 2.0, 5.0, 20.0}",NaN,NaN
2,Logistic Regression,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.1, 0.5, 0.05, 1.0, 5.0}","{0.0, 0.25, 0.5}"
3,Logistic Regression,Random shuffle,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.1, 1.0, 0.005, 0.01, 0.5, 0.05}","{0.0, 0.25, 1.0, 0.5}"
4,Linear SVM,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.05, 50.0, 0.1}",NaN
5,Linear SVM,Random shuffle,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.01, 0.001, 2.0, 0.005}",NaN


# Save processed tables for the plotting notebook

In [13]:
TASK = "hi"
DATASET = "kdr"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / TASK
    / DATASET
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Main protocol comparison tables
df_folds.to_csv(
    OUTPUT_DIR / "protocol_per_fold.csv",
    index=False,
)

summary_numeric.to_csv(
    OUTPUT_DIR / "protocol_summary_numeric.csv",
    index=False,
)

summary_table.to_csv(
    OUTPUT_DIR / "protocol_summary_display.csv",
    index=False,
)

delta_table.to_csv(
    OUTPUT_DIR / "protocol_delta.csv",
    index=False,
)

# Hyperparameter tables
hp_df.to_csv(
    OUTPUT_DIR / "hyperparameters_all.csv",
    index=False,
)

lr_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_lr.csv",
    index=False,
)

svm_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_svm.csv",
    index=False,
)

dt_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_dt.csv",
    index=False,
)

hp_set_summary.to_csv(
    OUTPUT_DIR / "hyperparameters_set_summary.csv",
    index=False,
)

print("Saved processed files in:")
print(OUTPUT_DIR)

print("\nFiles saved:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {file.name}")

Saved processed files in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/kdr

Files saved:
- hyperparameters_all.csv
- hyperparameters_dt.csv
- hyperparameters_lr.csv
- hyperparameters_set_summary.csv
- hyperparameters_svm.csv
- protocol_delta.csv
- protocol_per_fold.csv
- protocol_summary_display.csv
- protocol_summary_numeric.csv
